In [8]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import pygad

In [34]:
df = pd.read_csv('./dataset/TARP.csv')

# Define features and target
features = [col for col in df.columns if col in ['Soil Moisture', 'Temperature', 'Soil Humidity', 'Time']]
target = 'Status'

# Keep only features and target columns
columns_to_keep = features + [target]
df = df[columns_to_keep]

# Remove any NaN rows
df = df.dropna()

print(f"Features selected: {features}")
print(f"Number of features: {len(features)}")
print(f"Dataset shape after cleaning: {df.shape}")

X = df[features].values
y = df[target].values

df.head()

Features selected: ['Soil Moisture', 'Temperature', 'Soil Humidity', 'Time']
Number of features: 4
Dataset shape after cleaning: (100000, 5)


,Soil Moisture,Temperature,Soil Humidity,Time,Status
0,54,22,70,21,ON
1,12,20,40,104,OFF
2,34,26,35,62,ON
3,7,44,44,93,OFF
4,50,38,23,92,OFF


In [38]:
def init_mf_from_feature(values):
    min_val = np.min(values)
    max_val = np.max(values)

    # Use KMeans to find natural centers
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    kmeans.fit(values.reshape(-1, 1))
    centers = sorted(kmeans.cluster_centers_.flatten())
    c1, c2, c3 = centers

    LOW  = [min_val, c1, c2]
    MED  = [c1,      c2, c3]
    HIGH = [c2,      c3, max_val]

    return LOW + MED + HIGH  # 9 genes per feature

In [41]:
# Build base chromosome from data
base_chromosome = []
for i in range(X.shape[1]):
    genes = init_mf_from_feature(X[:, i])
    base_chromosome.extend(genes)

base_chromosome = np.array(base_chromosome)